In [ ]:
from IPython.display import HTML
display(HTML("<style>.rendered_html { font-size: 1.3em; } .code_cell .input_area { font-size: 1.1em; }</style>"))

# 2.4 Tune Regularization Hyperparameters
## Model Cycle Step 5: **Improve**
- Use GridSearchCV to find optimal C
- Tune ElasticNet's l1_ratio
- Select the best model
- Final evaluation on test set

## The Model Cycle
### 1. Build
### 2. Train
### 3. Predict
### 4. Evaluate
### **5. Improve** ← We are here

## What is GridSearchCV?
- Systematically tries many hyperparameter combinations
- Cross-validates each combination
- Picks the combination with the best average score

**Why not manual tuning?**
- Exhaustive search — doesn't miss anything
- Cross-validation prevents overfitting to validation data
- Tracks all results automatically

## What We're Tuning
| Parameter | Description | Range |
|:----------|:------------|:------|
| `C` | Inverse of regularization strength | 0.001 to 100 (log scale) |
| `l1_ratio` | ElasticNet mixing (0=L2, 1=L1) | 0.0 to 1.0 |

- **Small C** = strong regularization (simpler, may underfit)
- **Large C** = weak regularization (complex, may overfit)

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import pickle
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    classification_report, confusion_matrix, make_scorer
)

pd.options.display.max_columns = None

In [ ]:
data_filepath = '/content/drive/MyDrive/IR ML Cert/MLCert Course 3/Course 3 Data/'
models_filepath = '/content/drive/MyDrive/IR ML Cert/MLCert Course 3/Course 3 Data/'

df_training = pd.read_csv(f'{data_filepath}training.csv')
df_testing = pd.read_csv(f'{data_filepath}testing.csv')

X_train = df_training
y_train = df_training['SEM_3_STATUS']
X_test = df_testing
y_test = df_testing['SEM_3_STATUS']

print(f"Training data: {X_train.shape[0]} samples")
print(f"Test data: {X_test.shape[0]} samples")

In [ ]:
l2_model = pickle.load(open(f'{models_filepath}l2_ridge_logistic_model.pkl', 'rb'))
l1_model = pickle.load(open(f'{models_filepath}l1_lasso_logistic_model.pkl', 'rb'))
elasticnet_model = pickle.load(open(f'{models_filepath}elasticnet_logistic_model.pkl', 'rb'))
print("Models loaded successfully!")

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_scorer = make_scorer(f1_score, pos_label='N')
C_values = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
print("Cross-validation: 5-fold stratified")
print("Optimization metric: F1 score (positive class = 'N')")
print(f"C values to search: {C_values}")

## Tune L2 (Ridge)

In [ ]:
print("Tuning L2 (Ridge) model...")
l2_grid_search = GridSearchCV(
    estimator=l2_model,
    param_grid={'classifier__C': C_values},
    cv=cv, scoring=f1_scorer,
    return_train_score=True, n_jobs=-1, verbose=1
)
l2_grid_search.fit(X_train, y_train)
print(f"\nBest C: {l2_grid_search.best_params_['classifier__C']}")
print(f"Best CV F1: {l2_grid_search.best_score_:.4f}")

In [ ]:
l2_results = pd.DataFrame(l2_grid_search.cv_results_)
l2_results_display = l2_results[['param_classifier__C', 'mean_train_score', 'mean_test_score', 'std_test_score', 'rank_test_score']].sort_values('rank_test_score')
l2_results_display.columns = ['C', 'Train F1 (Mean)', 'CV F1 (Mean)', 'CV F1 (Std)', 'Rank']
print("L2 (Ridge) GridSearch Results:")
display(l2_results_display)

## Tune L1 (Lasso)

In [ ]:
print("Tuning L1 (Lasso) model...")
l1_grid_search = GridSearchCV(
    estimator=l1_model,
    param_grid={'classifier__C': C_values},
    cv=cv, scoring=f1_scorer,
    return_train_score=True, n_jobs=-1, verbose=1
)
l1_grid_search.fit(X_train, y_train)
print(f"\nBest C: {l1_grid_search.best_params_['classifier__C']}")
print(f"Best CV F1: {l1_grid_search.best_score_:.4f}")

In [ ]:
l1_results = pd.DataFrame(l1_grid_search.cv_results_)
l1_results_display = l1_results[['param_classifier__C', 'mean_train_score', 'mean_test_score', 'std_test_score', 'rank_test_score']].sort_values('rank_test_score')
l1_results_display.columns = ['C', 'Train F1 (Mean)', 'CV F1 (Mean)', 'CV F1 (Std)', 'Rank']
print("L1 (Lasso) GridSearch Results:")
display(l1_results_display)

## Tune ElasticNet
- Two parameters: C **and** l1_ratio
- 2D grid search

In [ ]:
l1_ratio_values = [0.1, 0.3, 0.5, 0.7, 0.9]
print("Tuning ElasticNet model...")
print(f"C values: {C_values}")
print(f"l1_ratio values: {l1_ratio_values}")
print(f"Total combinations: {len(C_values) * len(l1_ratio_values)}")

elasticnet_grid_search = GridSearchCV(
    estimator=elasticnet_model,
    param_grid={'classifier__C': C_values, 'classifier__l1_ratio': l1_ratio_values},
    cv=cv, scoring=f1_scorer,
    return_train_score=True, n_jobs=-1, verbose=1
)
elasticnet_grid_search.fit(X_train, y_train)
print(f"\nBest C: {elasticnet_grid_search.best_params_['classifier__C']}")
print(f"Best l1_ratio: {elasticnet_grid_search.best_params_['classifier__l1_ratio']}")
print(f"Best CV F1: {elasticnet_grid_search.best_score_:.4f}")

In [ ]:
elasticnet_results = pd.DataFrame(elasticnet_grid_search.cv_results_)
elasticnet_results_display = elasticnet_results[['param_classifier__C', 'param_classifier__l1_ratio', 'mean_train_score', 'mean_test_score', 'std_test_score', 'rank_test_score']].sort_values('rank_test_score').head(10)
elasticnet_results_display.columns = ['C', 'l1_ratio', 'Train F1 (Mean)', 'CV F1 (Mean)', 'CV F1 (Std)', 'Rank']
print("ElasticNet GridSearch Results (Top 10):")
display(elasticnet_results_display)

## Visualize the Hyperparameter Grid

In [ ]:
elasticnet_pivot = elasticnet_results.pivot_table(
    values='mean_test_score',
    index='param_classifier__l1_ratio',
    columns='param_classifier__C'
)
fig = px.imshow(elasticnet_pivot,
    labels=dict(x='C (Regularization Strength)', y='l1_ratio', color='CV F1 Score'),
    x=[str(c) for c in C_values], y=[str(r) for r in l1_ratio_values],
    color_continuous_scale='Viridis', aspect='auto',
    title='ElasticNet Hyperparameter Grid: CV F1 Scores')
fig.update_layout(height=400)
fig.show()

## Compare All Tuned Models

In [ ]:
comparison_results = [
    {'Model': 'L2 (Ridge)', 'Best C': l2_grid_search.best_params_['classifier__C'],
     'Best l1_ratio': 'N/A', 'CV F1 (Mean)': l2_grid_search.best_score_,
     'CV F1 (Std)': l2_results.loc[l2_results['rank_test_score']==1, 'std_test_score'].values[0]},
    {'Model': 'L1 (Lasso)', 'Best C': l1_grid_search.best_params_['classifier__C'],
     'Best l1_ratio': 'N/A', 'CV F1 (Mean)': l1_grid_search.best_score_,
     'CV F1 (Std)': l1_results.loc[l1_results['rank_test_score']==1, 'std_test_score'].values[0]},
    {'Model': 'ElasticNet', 'Best C': elasticnet_grid_search.best_params_['classifier__C'],
     'Best l1_ratio': elasticnet_grid_search.best_params_['classifier__l1_ratio'],
     'CV F1 (Mean)': elasticnet_grid_search.best_score_,
     'CV F1 (Std)': elasticnet_results.loc[elasticnet_results['rank_test_score']==1, 'std_test_score'].values[0]}
]

comparison_df = pd.DataFrame(comparison_results)
print("Best Tuned Models Comparison:")
display(comparison_df)

In [ ]:
fig = go.Figure()
fig.add_trace(go.Bar(x=comparison_df['Model'], y=comparison_df['CV F1 (Mean)'],
    error_y=dict(type='data', array=comparison_df['CV F1 (Std)']),
    marker_color=['#1f77b4', '#ff7f0e', '#2ca02c']))
fig.update_layout(title='Comparison of Tuned Models: CV F1 Scores',
    xaxis_title='Model', yaxis_title='CV F1 Score (Mean ± Std)', height=400)
fig.show()

## Select the Best Model

In [ ]:
best_idx = comparison_df['CV F1 (Mean)'].idxmax()
best_model_name = comparison_df.loc[best_idx, 'Model']

if best_model_name == 'L2 (Ridge)':
    best_model = l2_grid_search.best_estimator_
elif best_model_name == 'L1 (Lasso)':
    best_model = l1_grid_search.best_estimator_
else:
    best_model = elasticnet_grid_search.best_estimator_

print(f"Selected Best Model: {best_model_name}")
print(f"Best C: {comparison_df.loc[best_idx, 'Best C']}")
print(f"Best l1_ratio: {comparison_df.loc[best_idx, 'Best l1_ratio']}")
print(f"CV F1: {comparison_df.loc[best_idx, 'CV F1 (Mean)']:.4f}")

## Final Evaluation: Test Set
- We only evaluate on test set **ONCE** — after all tuning is complete
- This gives an unbiased estimate of real-world performance

In [ ]:
y_pred = best_model.predict(X_test)

test_f1 = f1_score(y_test, y_pred, pos_label='N')
test_precision = precision_score(y_test, y_pred, pos_label='N')
test_recall = recall_score(y_test, y_pred, pos_label='N')
test_accuracy = accuracy_score(y_test, y_pred)

print("="*50)
print(f"FINAL TEST SET EVALUATION: {best_model_name}")
print("="*50)
print(f"\nMinority Class ('N' = Students who leave):")
print(f"  F1 Score:    {test_f1:.4f}")
print(f"  Precision:   {test_precision:.4f}")
print(f"  Recall:      {test_recall:.4f}")
print(f"\nOverall Accuracy: {test_accuracy:.4f}")

In [ ]:
print("\nFull Classification Report:")
print(classification_report(y_test, y_pred))

## Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=['N', 'Y'])
fig = px.imshow(cm,
    labels=dict(x='Predicted', y='Actual', color='Count'),
    x=['Left (N)', 'Stayed (Y)'], y=['Left (N)', 'Stayed (Y)'],
    color_continuous_scale='Blues', text_auto=True, aspect='equal',
    title=f'Confusion Matrix: {best_model_name} on Test Set')
fig.update_layout(height=400)
fig.show()

In [ ]:
tn, fp, fn, tp = cm.ravel()
print("Confusion Matrix Interpretation:")
print(f"\n  True Positives (correctly identified departures):  {tp}")
print(f"  True Negatives (correctly identified stayers):     {tn}")
print(f"  False Positives (predicted departure, but stayed): {fp}")
print(f"  False Negatives (predicted stay, but left):        {fn}")

## How Does C Affect Performance?

In [ ]:
fig = make_subplots(rows=1, cols=3, subplot_titles=('L2 (Ridge)', 'L1 (Lasso)', 'ElasticNet (best l1_ratio)'))

fig.add_trace(go.Scatter(x=np.log10(l2_results['param_classifier__C'].astype(float)),
    y=l2_results['mean_test_score'], mode='lines+markers',
    error_y=dict(type='data', array=l2_results['std_test_score']),
    name='L2', line=dict(color='#1f77b4')), row=1, col=1)

fig.add_trace(go.Scatter(x=np.log10(l1_results['param_classifier__C'].astype(float)),
    y=l1_results['mean_test_score'], mode='lines+markers',
    error_y=dict(type='data', array=l1_results['std_test_score']),
    name='L1', line=dict(color='#ff7f0e')), row=1, col=2)

elasticnet_best_per_c = elasticnet_results.loc[
    elasticnet_results.groupby('param_classifier__C')['mean_test_score'].idxmax()]
fig.add_trace(go.Scatter(x=np.log10(elasticnet_best_per_c['param_classifier__C'].astype(float)),
    y=elasticnet_best_per_c['mean_test_score'], mode='lines+markers',
    error_y=dict(type='data', array=elasticnet_best_per_c['std_test_score']),
    name='ElasticNet', line=dict(color='#2ca02c')), row=1, col=3)

fig.update_xaxes(title_text='log10(C)')
fig.update_yaxes(title_text='CV F1 Score', row=1, col=1)
fig.update_layout(height=400, title_text='Effect of C on Model Performance', showlegend=False)
fig.show()

## Save Best Model

In [ ]:
best_model_filename = best_model_name.lower().replace(' ', '_').replace('(', '').replace(')', '')
save_path = f'{models_filepath}{best_model_filename}_tuned.pkl'
pickle.dump(best_model, open(save_path, 'wb'))
print(f"Best model saved to: {save_path}")

grid_search_results = {
    'l2_grid_search': l2_grid_search,
    'l1_grid_search': l1_grid_search,
    'elasticnet_grid_search': elasticnet_grid_search,
    'best_model_name': best_model_name
}
pickle.dump(grid_search_results, open(f'{models_filepath}grid_search_results.pkl', 'wb'))
print(f"Grid search results saved.")

## Key Takeaways
- **GridSearchCV** systematically finds optimal hyperparameters
- **C** controls regularization strength (small C = strong regularization)
- **ElasticNet l1_ratio** balances L1 vs L2 penalty
- Test set evaluation should only be done **once** after all tuning

## Module 2 Complete!
You've learned to:
1. Understand regularization concepts (2.1)
2. Build regularized models (2.2)
3. Train and compare models (2.3)
4. Tune hyperparameters (2.4)

**Next: Module 3 — Tree-Based Models**